---

### <center> 기본 세팅

---

In [1]:
import os 
import sys
import requests
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns   
import math
import scipy.stats as stats
from io import StringIO
from datetime import datetime, timedelta

import warnings 
warnings.filterwarnings("ignore")

In [36]:
# 폰트 깨짐 방지
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic') 
else:
    
    plt.rc('font', family='NanumGothic')

# 마이너스 기호방지
plt.rcParams['axes.unicode_minus'] = False

In [2]:
# 데이터 불러오기 (용량 최적화를 위해 dtype 지정)
articles = pd.read_csv('../Data Folder/H&M dataset/articles.csv', dtype={'article_id': str})
customers = pd.read_csv('../Data Folder\H&M dataset\customers.csv')
transactions = pd.read_csv('../Data Folder/H&M dataset/transactions_train.csv', dtype={'article_id': str})
sample_sumbission = pd.read_csv('../Data Folder/H&M dataset/sample_submission.csv')

---

### <center> 1. 기초 탐색 

---

In [ ]:
# 판매채널 (1:오프라인 2:온라인)
# transactions_train_df['sales_channel_id'].value_counts()

In [ ]:
# 카테고리 분포
# f, ax = plt.subplots(figsize=(15, 7))
# ax = sns.histplot(data=articles_df, y='index_name', color='orange')
# ax.set_xlabel('count by index name')
# ax.set_ylabel('index name')
# plt.show()

In [ ]:
# f, ax = plt.subplots(figsize=(15, 7))
# ax = sns.histplot(data=articles_df, y='garment_group_name', color='orange', hue='index_group_name', multiple="stack")
# ax.set_xlabel('의류 그룹(종류)별 수량(개수)')
# ax.set_ylabel('의류 그룹(종류)')
# plt.show()

In [ ]:
# customers_df['age'].mean()

In [ ]:
# 상품 그룹(종류)별 수량
# articles_df.groupby(['index_group_name', 'index_name']).count()['article_id']

In [ ]:
# 상품 그룹,종류별 수량
#pd.options.display.max_rows = None
#articles_df.groupby(['product_group_name', 'product_type_name']).count()['article_id']

---

### <center> 2. 전처리

---

In [39]:
# 날짜 타입 변환
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])

In [40]:
# customers 결측치 확인
customers.isna().sum()

## FN, Active, club_member_status, fashion_news_frequency, age에 결측치 발견

customer_id                    0
FN                        895050
Active                    907576
club_member_status          6062
fashion_news_frequency     16011
age                        15861
postal_code                    0
dtype: int64

In [41]:
# articles 결측치 확인
articles.isna().sum()

## detail_desc에 하나 발견

article_id                        0
product_code                      0
prod_name                         0
product_type_no                   0
product_type_name                 0
product_group_name                0
graphical_appearance_no           0
graphical_appearance_name         0
colour_group_code                 0
colour_group_name                 0
perceived_colour_value_id         0
perceived_colour_value_name       0
perceived_colour_master_id        0
perceived_colour_master_name      0
department_no                     0
department_name                   0
index_code                        0
index_name                        0
index_group_no                    0
index_group_name                  0
section_no                        0
section_name                      0
garment_group_no                  0
garment_group_name                0
detail_desc                     416
dtype: int64

In [42]:
# transactions 결측치 확인
transactions.isna().sum()

t_dat               0
customer_id         0
article_id          0
price               0
sales_channel_id    0
dtype: int64

In [43]:
# sample_sumbission 결측치 확인
sample_sumbission.isna().sum()

customer_id    0
prediction     0
dtype: int64

이커머스 데이터에서 결측치는 정보의 부재가 아니라 '해당 사항 없음'인 경우가 많습니다.

- Customers: 
    - FN, Active 열의 결측치는 활동하지 않는 유저이므로 0으로 채웁니다. 
    - fashion_news_frequency는 None으로 대체합니다.
    - club_member_status는 NONE으로 치환합니다.

- Articles: 상품 설명(detail_desc) 컬럼 삭제
    - 
- Transactions: price의 이상치(너무 비싸거나 싼 값)를 제거합니다. (팝업스토어는 대중적인 가격대가 중요)
    - 

In [44]:
customers['club_member_status'].value_counts()

club_member_status
ACTIVE        1272491
PRE-CREATE      92960
LEFT CLUB         467
Name: count, dtype: int64

In [ ]:
customers['fashion_news_frequency'].value_counts()

# Regularly # 일주일 수신
# monthly # 월별 수신
# NONE  # 수신 하지않음

## 이 컬럼으로 할 수 있는 분석은?
# 고객별 수신종류 리텐션, 코호트 분석 
# 결측치는 'NONE' 치환

fashion_news_frequency
NONE         877711
Regularly    477416
Monthly         842
Name: count, dtype: int64

In [ ]:
customers['FN'].value_counts()

# 0일 경우는 수신하지않음 1은 수신

## 이 컬럼으로 할 수 있는 분석은?
# FN이 1로 수신상태인데 Fashion_news_frequency가 Regularly나 monthly로 되어 있는 고객층은 1로 변환
# 결측치는 0으로 대체

FN
1.0    476930
Name: count, dtype: int64

In [ ]:
# transactions price *590
# price 컬럼은 기본적으로 590을 곱해야 현실적인 가격이 측정됨
transactions['price'] = transactions['price'] 

In [ ]:
customers['Active'].value_counts()

# 활동상태를 뜻하는 Active 
# 1일 경우는 활동 / null은 활동하지않음 
## 결측치는 0으로 치환 -> 결측치는 활동하지 않는다는 의미로 가정

Active
1.0    464404
Name: count, dtype: int64

In [ ]:
customers['club_member_status'].isna().sum()

# 멤버쉽 상태를 뜻함
# Active는 클럽 가입 회원 / Pre_create 가입 절차중인 예비회원 / LEFT CLUB 탈퇴한 회원
## 멤버쉽을 가입하지 않은 고객군으로 가정 'NONE' 으로 치환

np.int64(6062)

In [ ]:
customers['age'].isna().sum()

# 고객 연령를 뜻하는 age
## 결측치는 찬혁님 아이디어는 해당 연령대가 많이 산 제품군으로 채워넣자
## 추정하기엔 위험하니 결측치 삭제 

np.int64(15861)

In [ ]:
## customers 결측치 처리 
customers['FN'] = customers['FN'].fillna(0) # FN은 0
customers['Active'] = customers['Active'].fillna(0) # Active 결측치 0으로 치환
customers['age'] = customers['age'].fillna(customers['age'].median()) # 연령은 중앙값으로 설정 (또는 평균)
customers['club_member_status'] = customers['club_member_status'].fillna('NONE') # ACTIVE, PRE-CREATE 등 기존 상태는 유지하되, 결측치는 LEFT 또는 NONE으로 치환
customers['fashion_news_frequency'] = customers['fashion_news_frequency'].replace('None', 'NONE') # Regularly, Monthly는 유지하고, 결측치와 None은 통합하여 NONE으로 처리.
customers['fashion_news_frequency'] = customers['fashion_news_frequency'].fillna('NONE') # ''

# 처리확인
customers.isna().sum()

customer_id               0
FN                        0
Active                    0
club_member_status        0
fashion_news_frequency    0
age                       0
postal_code               0
dtype: int64

In [ ]:
# transactions 이상치 처리

q_high = transactions['price'].quantile(0.99) # 상위 99% 데이터를 제외하고 드랍
transactions = transactions[transactions['price'] <= q_high]

transactions.isna().sum()

t_dat               0
customer_id         0
article_id          0
price               0
sales_channel_id    0
dtype: int64

In [ ]:
# price 컬럼의 값을 *590
# 실제 달러 가격과 매칭 
transactions['price'] = transactions['price']

In [ ]:
# age 컬럼 정수형으로 변환
customers['age'] = customers['age'].dtype('')

In [ ]:
# age 연령대 이상치처리
# 팝업스토어에 80세 이상은 숫자도 적고 구매력이 없기때문에 이상치로 분류
customers['age'].max()

np.float64(99.0)

In [ ]:
# 1. Age (나이): 실수에서 정수로 변환
customers['age'] = customers['age'].astype(int)

# 2. 범주형 데이터 (Categorical): 메모리 절약의 핵심
# 'club_member_status'나 'fashion_news_frequency'처럼 반복되는 글자는
# 'category' 타입으로 바꾸면 메모리 사용량이 80% 이상 줄어듭니다.
customers['club_member_status'] = customers['club_member_status'].astype('category')
customers['fashion_news_frequency'] = customers['fashion_news_frequency'].astype('category')

# 3. ID 칼럼: 문자열로 통일
# 앞서 겪으셨던 '0'이 잘리는 문제를 방지하기 위해 문자열로 고정합니다.
articles['article_id'] = articles['article_id'].astype(str)
articles['customer_id'] = articles['customer_id'].astype(str)

# 4. 날짜 데이터: 문자열에서 날짜형으로 변환 (분석 필수)
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])


customers['FN'] = customers['FN'].astype(int)
customers['Active'] = customers['Active'].astype(int)

In [3]:
transactions['sales_channel_id'].value_counts()

sales_channel_id
2    22379862
1     9408462
Name: count, dtype: int64

---

### <center> 3. 피처 엔지니어링 

---

- Offline_Affinity (오프라인 친밀도): `sales_channel_id` 가 1이면 오프라인, 2면 온라인 /  오프라인 구매 비중이 높은 유저를 찾아 팝업 타겟팅 우선순위를 부여하는 피처

- Style_Persona (스타일 페르소나): `index_group_name`이나 `perceived_colour_master_name` 을 활용해 유저가 '무채색 모던'파인지 '비비드 캐주얼'파인지 분류하는 피처

In [ ]:
# 고객별 총 구매액 및 선호 채널 용도 DF
customer_stats = transactions.groupby('customer_id').agg({
    'price': 'sum', # 총 구매액
    'sales_channel_id': lambda x: (x == 1).mean(), # 오프라인 구매 비중 계산 (1이면 OFF / 2면 ON / == 1의 비율확인)
    'article_id': 'count' # 총 구매 횟수
}).rename(columns={'price': 'total_spend', 'sales_channel_id': 'offline_ratio', 'article_id': 'purchase_count'})

# 상권 매핑을 위한 고객 연령대 그룹화 --> 팝업 타겟팅용도 --> 
customers['age_group'] = pd.cut(customers['age'], bins=[0, 19, 29, 39, 49, 99], labels=['Teen', '20s', '30s', '40s', '50+'])

In [ ]:
customers['age_group'].value_counts()

age_group
20s     528358
50+     317992
30s     234068
40s     204118
Teen     71583
Name: count, dtype: int64

- User Segmentation: 구매 금액($Monetary$), 구매 빈도($Frequency$), 오프라인 선호도 기반 타겟 그룹화.